# 可选实验：Softmax Function

本实验将探索 softmax function。它用于 softmax regression，也用于 neural network 的 multiclass classification。

<figure style="text-align:center;">
  <img src="./images/C2_W2_Softmax_Header.PNG" alt="Softmax 课程图示" width="600">
</figure>


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from IPython.display import display, Markdown, Latex
from sklearn.datasets import make_blobs
%matplotlib widget
from matplotlib.widgets import Slider
from lab_utils_common import dlc
from lab_utils_softmax import plt_softmax
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

> **编号说明：** 本课程的 Notebook 通常从 0 到 $N-1$ 编号，即 $\sum_{i=0}^{N-1}$；课程视频通常从 1 到 $N$，即 $\sum_{i=1}^{N}$。前者符合代码习惯，后者使公式更简洁。本实验公式较多，因此采用从 1 到 $N$ 的数学编号。


## Softmax Function

Softmax regression 和使用 softmax 输出的 neural network 都会生成 $N$ 个输出，并从中选择预测类别。首先由线性计算得到 vector $\mathbf{z}$，再通过 softmax 转换为 probability distribution（概率分布）。

Softmax 的每个输出位于 0 与 1 之间，所有输出之和为 1，因此可以解释为各个类别的概率。较大的输入对应较大的输出概率。

<figure style="text-align:center;">
  <img src="./images/C2_W2_SoftmaxReg_NN.png" alt="Softmax regression 与 neural network" width="600">
</figure>


Softmax 定义为：

$$
a_j=\frac{e^{z_j}}{\sum_{k=1}^{N}e^{z_k}} \tag{1}
$$

输出 $\mathbf{a}$ 是长度为 $N$ 的 vector。对于 softmax regression，可以写成：

$$
\mathbf{a}(\mathbf{x})=
\begin{bmatrix}
P(y=1\mid\mathbf{x};\mathbf{W},\mathbf{b})\\
\vdots\\
P(y=N\mid\mathbf{x};\mathbf{W},\mathbf{b})
\end{bmatrix}
=\frac{1}{\sum_{k=1}^{N}e^{z_k}}
\begin{bmatrix}
e^{z_1}\\\vdots\\e^{z_N}
\end{bmatrix}
\tag{2}
$$


因此，输出是一个概率 vector。第一个元素表示：给定输入 $\mathbf{x}$ 和模型参数，样本属于第一个类别的概率。

下面使用 NumPy 实现。


In [ ]:
def my_softmax(z):
    ez = np.exp(z)              #element-wise exponenial
    sm = ez/np.sum(ez)
    return(sm)

通过下方滑块改变各个 `z` 输入值。


In [ ]:
plt.close("all")
plt_softmax(my_softmax)

改变 `z` 时，请观察：

- 分子中的 exponential（指数）会放大输入值之间的差异。
- 所有输出之和始终为 1。
- Softmax 同时作用于全部输出。例如，改变 `z0` 会影响 `a0` 到 `a3`。ReLU 和 sigmoid 则是逐个输入独立计算输出。


## Cost

<figure style="text-align:center;">
  <img src="./images/C2_W2_SoftMaxCost.png" alt="Softmax 对应的 cross-entropy loss" width="400">
</figure>


Softmax 常与 cross-entropy loss（交叉熵损失）配合：

$$
L(\mathbf{a},y)=
\begin{cases}
-\log(a_1), & y=1,\\
\vdots & \vdots\\
-\log(a_N), & y=N.
\end{cases}
\tag{3}
$$

$y$ 是该样本的 target 类别，$\mathbf{a}$ 是 softmax 输出，其各元素为概率，总和为 1。

> **回顾：** Loss 针对单个样本，cost 汇总全部样本。

公式 (3) 只选取真实类别对应的概率。可使用 indicator function（指示函数）表示这一选择：

$$
\mathbf{1}\{y=n\}=
\begin{cases}
1, & y=n,\\
0, & y\ne n.
\end{cases}
$$

因此，所有样本的平均 cost 为：

$$
J(\mathbf{W},\mathbf{b})=-\frac{1}{m}\sum_{i=1}^{m}\sum_{j=1}^{N}
\mathbf{1}\{y^{(i)}=j\}\log\left(\frac{e^{z_j^{(i)}}}{\sum_{k=1}^{N}e^{z_k^{(i)}}}\right)
\tag{4}
$$

其中，$m$ 是样本数，$N$ 是输出类别数。


## TensorFlow

本实验比较 TensorFlow 中两种实现 softmax 与 cross-entropy loss 的方式：直接在输出层使用 softmax，以及训练时将 softmax 合并到 loss 中。前者更直观，后者通常具有更好的 numerical stability。

先创建用于训练 multiclass classification 模型的数据集。


In [ ]:
# make  dataset for example
centers = [[-5, 2], [-2, -2], [1, 2], [5, -2]]
X_train, y_train = make_blobs(n_samples=2000, centers=centers, cluster_std=1.0,random_state=30)

### 直接在输出层使用 Softmax


下面的模型在最后一个 `Dense` layer 中使用 softmax activation，再通过 `compile` 单独指定 loss function。

这里使用 `SparseCategoricalCrossentropy`，对应公式 (3)。由于 softmax 已在最后一层完成，loss function 接收的是概率 vector。


In [ ]:
model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'softmax')    # < softmax activation here
    ]
)
model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(0.001),
)

model.fit(
    X_train,y_train,
    epochs=10
)
        

由于 softmax 已包含在输出层中，模型输出就是概率 vector。


In [ ]:
p_nonpreferred = model.predict(X_train)
print(p_nonpreferred [:2])
print("largest value", np.max(p_nonpreferred), "smallest value", np.min(p_nonpreferred))

### 更稳定的实现方式

<img align="right" src="./images/C2_W2_softmax_accurate.png" alt="将 softmax 与 loss 合并以改善数值稳定性" style="width:400px;padding:10px 20px;">

如课程所述，训练时合并 softmax 与 loss 的计算，可以得到更稳定、准确的数值结果。下面采用这一实现方式。


在这一实现中，最后一层使用 linear activation，输出称为 **logits**。

Loss function 设置 `from_logits=True`，表示输入尚未经过 softmax，需要在 loss 计算内部处理。这样便于使用数值上更稳定的联合实现。


In [ ]:
preferred_model = Sequential(
    [ 
        Dense(25, activation = 'relu'),
        Dense(15, activation = 'relu'),
        Dense(4, activation = 'linear')   #<-- Note
    ]
)
preferred_model.compile(
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),  #<-- Note
    optimizer=tf.keras.optimizers.Adam(0.001),
)

preferred_model.fit(
    X_train,y_train,
    epochs=10
)
        

#### 处理输出

这个模型输出的是 logits，可能是较大的负数或正数。若需要概率预测，必须再通过 softmax 转换。先观察原始输出：


In [ ]:
p_preferred = preferred_model.predict(X_train)
print(f"two example output vectors:\n {p_preferred[:2]}")
print("largest value", np.max(p_preferred), "smallest value", np.min(p_preferred))

这些输出还不是概率！如果需要概率，应使用 [softmax](https://www.tensorflow.org/api_docs/python/tf/nn/softmax) 进行转换。


In [ ]:
sm_preferred = tf.nn.softmax(p_preferred).numpy()
print(f"two example output vectors:\n {sm_preferred[:2]}")
print("largest value", np.max(sm_preferred), "smallest value", np.min(sm_preferred))

如果只需要选择最可能的类别，则无需先计算 softmax。直接使用 [`np.argmax()`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html) 找出最大输出对应的 index 即可。


In [ ]:
for i in range(5):
    print( f"{p_preferred[i]}, category: {np.argmax(p_preferred[i])}")

## SparseCategoricalCrossentropy 与 CategoricalCrossentropy

TensorFlow 支持两种 target 表示，所选 loss 决定预期格式：

- **SparseCategoricalCrossentropy**：target 是类别 index 对应的整数。例如，10 个类别用 0–9 表示。
- **CategoricalCrossentropy**：target 使用 one-hot encoding，真实类别对应位置为 1，其余位置为 0。例如，10 类中的类别 2 表示为 `[0,0,1,0,0,0,0,0,0,0]`。


## 恭喜完成实验！

你已经：

- 熟悉 softmax 及其在 softmax regression、neural network 中的作用。
- 学习了训练时更稳定的组合：输出层使用 linear activation，loss 使用 `SparseCategoricalCrossentropy(from_logits=True)`。
- 理解了 softmax 与 ReLU、sigmoid 的区别：它同时联系多个输出。
